# Pocket-Agent — Training Notebook
**Run this notebook on Colab T4 GPU (Runtime → Change runtime type → T4 GPU)**

This notebook:
1. Installs dependencies
2. Clones the repo
3. Generates synthetic training data
4. Fine-tunes `Qwen2.5-0.5B-Instruct` with QLoRA
5. Quantizes to GGUF Q3_K_M
6. Evaluates on built-in test suite
7. Uploads artifacts to HuggingFace Hub

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('No GPU detected! Go to Runtime → Change runtime type → T4 GPU')
print(result.stdout[:500])

In [ ]:
# ── Cell 2: Install training dependencies ────────────────────────────────────
# This takes ~2 minutes
%pip install -q \
    'torch>=2.2.0' \
    'transformers>=4.44.0' \
    'peft>=0.12.0' \
    'trl>=0.12.0' \
    'datasets>=2.19.0' \
    'bitsandbytes>=0.43.0' \
    'accelerate>=0.29.0' \
    'huggingface_hub>=0.20.0'

print('Dependencies installed ✅')

In [ ]:
# ── Cell 3: Clone repo ────────────────────────────────────────────────────────
import os

REPO_URL = 'https://github.com/YOUR_USERNAME/pocket-agent'  # ← UPDATE THIS

if not os.path.exists('pocket-agent'):
    !git clone {REPO_URL}

%cd pocket-agent
print('Working directory:', os.getcwd())

In [ ]:
# ── Cell 4: Generate training data ───────────────────────────────────────────
!python src/data/generate.py
!python src/data/lint.py

# Verify
!wc -l data/train_clean.jsonl

In [ ]:
# ── Cell 5: Fine-tune with QLoRA (~25-35 minutes) ────────────────────────────
!python src/train/sft_lora.py

import os
files = os.listdir('artifacts/lora_adapter')
print('Adapter files:', files)

In [ ]:
# ── Cell 6: Build llama.cpp and quantize ─────────────────────────────────────
import os

# Clone llama.cpp
if not os.path.exists('llama.cpp'):
    !git clone https://github.com/ggml-org/llama.cpp --depth=1

# Install Python deps for conversion
%pip install -q gguf sentencepiece

# Build llama.cpp tools with CMake (~3 min)
!cd llama.cpp && cmake -B build -DCMAKE_BUILD_TYPE=Release && cmake --build build --config Release -j4

# Run full pipeline
!bash src/quantize/quantize.sh

In [ ]:
# ── Cell 7: Evaluate on built-in test suite ───────────────────────────────────
!python src/eval/score.py --verbose

# If score < 0.75, retrain one more epoch:
# !python src/train/sft_lora.py  (checkpoint resumes automatically)

In [ ]:
# ── Cell 8: Upload to HuggingFace Hub ────────────────────────────────────────
# Create a free account at https://huggingface.co then:
# 1. Go to Settings → Access Tokens → New token (write)
# 2. Paste the token below (or use Colab Secrets)

HF_TOKEN = ''          # ← PASTE YOUR HF WRITE TOKEN
HF_REPO  = ''          # ← e.g. 'yourusername/pocket-agent'

if not HF_TOKEN or not HF_REPO:
    print('⚠️  Set HF_TOKEN and HF_REPO above before running this cell')
else:
    from huggingface_hub import HfApi, login
    login(token=HF_TOKEN)
    api = HfApi()

    # Create repo
    api.create_repo(HF_REPO, repo_type='model', exist_ok=True)

    # Upload GGUF model
    print('Uploading model.gguf (~220MB)...')
    api.upload_file(
        path_or_fileobj='artifacts/model.gguf',
        path_in_repo='model.gguf',
        repo_id=HF_REPO,
        repo_type='model',
    )

    # Upload LoRA adapter
    print('Uploading lora_adapter/...')
    api.upload_folder(
        folder_path='artifacts/lora_adapter',
        path_in_repo='lora_adapter',
        repo_id=HF_REPO,
        repo_type='model',
    )

    print(f'\nDone! Model live at: https://huggingface.co/{HF_REPO}')
    print('Copy this URL into README.md and 02_judge_demo.ipynb')

In [ ]:
# ── Cell 9: Final gate checks before pushing ──────────────────────────────────
import os, time, sys
sys.path.insert(0, '.')

# Gate 1: No network imports
import subprocess
r = subprocess.run(
    ['grep', '-n', 'import requests\\|import urllib\\|import http\\|import socket', 'inference.py'],
    capture_output=True, text=True
)
print('Gate 1 - Network imports:', 'PASS ✅' if not r.stdout.strip() else f'FAIL ❌ {r.stdout}')

# Gate 2: Model size
size = os.path.getsize('artifacts/model.gguf') / 1e6
print(f'Gate 2 - Model size: {size:.0f}MB', '✅ <500MB' if size < 500 else '❌ too large')
print(f'         Bonus gate: {"✅ <250MB" if size < 250 else "❌ >250MB"}')

# Gate 3: Latency
from inference import run as _run
prompts = ['weather in Tokyo in C', '100 USD to EUR', 'convert 5 km to miles'] * 5
times = []
for p in prompts:
    t = time.time(); _run(p, []); times.append((time.time()-t)*1000)
mean_ms = sum(times)/len(times)
print(f'Gate 3 - Latency: mean={mean_ms:.1f}ms', '✅' if mean_ms < 200 else '❌ too slow')

# Gate 4: Adapter loads
from transformers import AutoModelForCausalLM
from peft import PeftModel
import torch
base = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct', torch_dtype=torch.float16, device_map='cpu')
PeftModel.from_pretrained(base, './artifacts/lora_adapter')
print('Gate 4 - Adapter loads: ✅')
del base